### This version is in development

Logs are still safe to be broken into chunks based on config_fname \
TIMESTAMP_STARTS Now indicates where the timestamp starts (as opposed to TIMESTAMP used in previous versions) 

Some new variables are present also, all variables that are present in log file order of appearance:
- `RETRY`
- `FILENAME`
- `STRING` (new; `string` in previous DB)
- `PORT` (new; `port` in previous DB)
- `OM_KEY` (new; not previously included)
- `ILLUMINATION` (new; `illum` in previous)
- `LED_ON`
- `LED_OFF`
- `CAMERA` (new; `cam` in previous)
- `CAPTURE_START`
- `CAPTURE_STOP`

Variables still not included in log file (scraped from .raw fname):
- `run_type`
- `device`
- `gain`
- `exposure`

This version of the fname scraper (below comment `#Scrape data from file name` in Cell 3) works with .raw fnames ending in `_date-time_trial?.raw`

I want to make names more consistent so **going forward everything will be uppercase and underscored.** This means that I will have to go back and alter previous versions of DB though.


In [11]:
import numpy as np
import uproot
import matplotlib as mpl
import os
import datetime as dt
%matplotlib inline
import pandas as pd
import matplotlib.pyplot as plt
import datetime
import matplotlib.dates as mdates
from datetime import datetime, timedelta
from itertools import islice



import json
from urllib.parse import urlencode
from urllib.request import urlopen, Request


In [12]:
# splits log file into chunks based on position of json config files
# first chunk is junk
# first line of chunk is path to json config file
# second line of chunk is the contents of the json file
# file path - path with lines of data
# chunks - array of arrays
def split_log_by_json(filepath):
    chunks = []
    current_chunk = []

    with open(filepath, "r") as f:
        for line in f:
            if ".json" in line:
                if current_chunk:
                    chunks.append(current_chunk)
                current_chunk = [line]
            else:
                current_chunk.append(line)

    if current_chunk:
        chunks.append(current_chunk)

    return chunks

In [13]:
format_pattern = '%Y-%m-%d %H:%M:%S.%f'

In [15]:
log_file = 'log87_88_89_Jan22.log' # Manually input

chunk_lst = split_log_by_json(log_file)
raw_keywords = ['DEVICE', 'GAIN', 'EXPOSURE']


result  = [] # Dictionary that will be transformed into .json

for i, chunk in enumerate(chunk_lst):
    if ".json" in chunk[0]: 
        json_fname = chunk[0].strip()
        json_lst = chunk[2].strip()
        contents = chunk[1:]
        for j, line in enumerate(contents):
            if "TIMESTAMP_STARTS" in line:
                # Scrape data from log file
                d = {}
                lst = contents[j+1:j+12] # Metadata in log file
                raw_fname = contents[j+2].split(":", 1)[1].strip() # path of .raw file
                for line in lst:
                    if ":" in line:
                        for k, v in [line.split(":", 1)]:
                            if k.strip() == 'ILLUMINATION':
                                v = v.strip().replace('LED_','')
                                d[k.strip()] = v
                            elif k.strip() == 'CAMERA':
                                v = v.strip().replace('CAM_','')
                            else:
                                d[k.strip()] = v.strip()
                
                # Scrape data from file name
                if raw_fname != "None":
                    raw_split = raw_fname.strip().split('_')
                    run_type = '_'.join(raw_split[1:-9])
                    d['RUN_TYPE'] = run_type
                    d['DEVICE'] = raw_split[-8].strip()
                    d['GAIN'] = raw_split[-4].strip().replace('gain','')
                    d['EXPOSURE'] = raw_split[-3].strip().replace('exposure','')
                else:
                    d['RUN_TYPE'] = "None"
                    for a in range(len(raw_keywords)):
                        d[raw_keywords[a]] = "None" 
                        
                    
                    
                d["CONFIG_FNAME"] = json_fname
                d["CONFIG_LST"] = json_lst
                result.append(d)

output_json_name = log_file.replace(".log", "_parsed.json")
with open(output_json_name, "w") as f:
    json.dump(result, f, indent=4)